## 4조 加油~~! 

### 데이터 전처리 입니다 ! 현재 raw 데이터 2개 파일이 있어요 



### 1. raw파일과 csv 파일 목록 확인 

In [3]:
from pathlib import Path

# 실제 프로젝트 폴더를 직접 지정
# 이 PROJECT_ROO 변수 안에 실제 프로젝트 폴더 경로를 저장  # 이건 내 컴퓨터 기준~ 
PROJECT_ROOT = Path(r"C:\dev\project\project_t4_V2")  

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 위치:", PROJECT_ROOT)
print("raw 폴더 위치:", RAW_DIR)
print()

if not RAW_DIR.exists():
    print("raw 폴더를 찾지 못했습니다.")
else:
    csv_files = sorted(RAW_DIR.glob("*.csv"))

    if not csv_files:
        print("raw 폴더에 CSV 파일이 없습니다.") 
    else:
        print("찾은 CSV 파일:")
        for number, file in enumerate(csv_files, start=1):       
            print(f"{number}. {file.name} / {file.stat().st_size:,} bytes")

프로젝트 위치: C:\dev\project\project_t4_V2
raw 폴더 위치: C:\dev\project\project_t4_V2\data\raw

찾은 CSV 파일:
1. 자동차제작결함신고정보.csv / 3,161,019 bytes
2. 차종별 리콜대수.csv / 3,999,666 bytes


In [4]:
import pandas as pd
from IPython.display import display

def read_csv_auto(file_path):
    # 한글 인코딩을 순서대로 시도합니다.
    for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            data = pd.read_csv(
                file_path,
                encoding=encoding,
                dtype=str
            )
            return data, encoding
        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        "파일 인코딩을 찾지 못했습니다."
    )

# 파일별로 읽고 일부 행만 출력
dataframes = {}

for file_path in csv_files:
    data, used_encoding = read_csv_auto(file_path)
    dataframes[file_path.name] = data

    print("=" * 60)
    print("파일명:", file_path.name)
    print("사용 인코딩:", used_encoding)
    print("행 수:", len(data))
    print("열 수:", len(data.columns))
    print("열 이름:", data.columns.tolist())
    print()
    print("앞에서 5개 행:")
    
    display(data.head(5))

파일명: 자동차제작결함신고정보.csv
사용 인코딩: utf-8-sig
행 수: 64807
열 수: 4
열 이름: ['접수일자', '제작사', '차명', '모델년도']

앞에서 5개 행:


,접수일자,제작사,차명,모델년도
0,2019-01-02,메르세데스-벤츠,GLA45 AMG 4Matic,2015
1,2019-01-02,현대자동차,i40 Saloon,2012
2,2019-01-02,르노코리아,SM5,2008
3,2019-01-02,BMW,BMW 530i,2018
4,2019-01-02,BMW,BMW 520d,2016


파일명: 차종별 리콜대수.csv
사용 인코딩: cp949
행 수: 13560
열 수: 7
열 이름: ['제작자', '차명', '생산기간(부터)', '생산기간(까지)', '리콜개시일', '리콜대수', '리콜사유']

앞에서 5개 행:


,제작자,차명,생산기간(부터),생산기간(까지),리콜개시일,리콜대수,리콜사유
0,벤츠,ML280 CDI,2006-04-18,2009-03-18,2012-01-09,529,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
1,벤츠,E220 CDI,2006-07-04,2009-01-15,2012-01-09,717,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
2,벤츠,S350 BLUETEC,2010-08-05,2011-02-02,2012-01-09,75,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
3,벤츠,ML300 CDI,2009-07-15,2011-04-06,2012-01-09,554,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
4,벤츠,C220 CDI,2007-01-03,2008-12-15,2012-01-09,549,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...


## 1. raw 폴더 확인과 CSV 불러오기

리콜 파일은 한글 Windows 인코딩일 수 있으므로 여러 인코딩을 순서대로 시도합니다.

In [ ]:
def read_csv_auto(path):
    # UTF-8과 CP949 중 실제 파일에 맞는 인코딩을 찾습니다.
    last_error = None
    for encoding in ['utf-8-sig', 'cp949', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except UnicodeDecodeError as error:
            last_error = error
    raise last_error

def find_raw_file(keyword):
    matches = [p for p in raw_files if keyword in p.stem]
    if not matches:
        raise FileNotFoundError(f'원본 파일을 찾지 못했습니다: {keyword}')
    return matches[0]

defect_path = find_raw_file('자동차제작결함신고정보')
recall_path = find_raw_file('차종별 리콜대수')
defect_raw, defect_encoding = read_csv_auto(defect_path)
recall_raw, recall_encoding = read_csv_auto(recall_path)

print('신고 파일:', defect_path.name, defect_encoding, defect_raw.shape)
print('리콜 파일:', recall_path.name, recall_encoding, recall_raw.shape)
display(defect_raw.head())
display(recall_raw.head())

## 2. 열 이름 확인 및 공통 이름으로 변경

신고 파일의 모델연도와 리콜 파일의 생산기간은 서로 다른 시간 기준입니다. 나중에 억지로 같은 연도로 JOIN하지 않고 각각 보존합니다.

In [ ]:
print('신고 원본 열:', defect_raw.columns.tolist())
print('리콜 원본 열:', recall_raw.columns.tolist())

defect = defect_raw.rename(columns={
    '접수일자': 'received_date',
    '제작사': 'manufacturer_name',
    '차명': 'model_name',
    '모델년도': 'model_year',
}).copy()

recalls = recall_raw.rename(columns={
    '제작사': 'manufacturer_name',
    '차명': 'model_name',
    '생산기간(부터)': 'production_start_date',
    '생산기간(까지)': 'production_end_date',
    '리콜개시일': 'recall_start_date',
    '리콜대수': 'affected_count',
    '리콜사유': 'recall_reason',
}).copy()

print('정리된 신고 열:', defect.columns.tolist())
print('정리된 리콜 열:', recalls.columns.tolist())

In [ ]:
## 3. 기본 형식 정리

def clean_text_columns(frame, columns):
    frame = frame.copy()
    for column in columns:
        if column in frame.columns:
            frame[column] = (
                frame[column].astype('string')
                .str.replace(r'\s+', ' ', regex=True)
                .str.strip()
                .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
            )
    return frame

defect = clean_text_columns(defect, ['manufacturer_name', 'model_name', 'received_date'])
recalls = clean_text_columns(recalls, ['manufacturer_name', 'model_name', 'production_start_date', 'production_end_date', 'recall_start_date', 'recall_reason'])
defect['model_year'] = pd.to_numeric(defect['model_year'], errors='coerce').astype('Int64')
recalls['affected_count'] = pd.to_numeric(recalls['affected_count'], errors='coerce').astype('Int64')

# 원본 행 번호는 원본 자료를 다시 찾기 위한 추적용 값입니다. 신고 행은 중복 제거하지 않습니다.
defect.insert(0, 'source_row_no', range(2, len(defect) + 2))
recalls.insert(0, 'source_row_no', range(2, len(recalls) + 2))
defect['source_file'] = defect_path.name
recalls['source_file'] = recall_path.name
display(defect.head())
display(recalls.head())

## 4. 제외 목록을 확인한 뒤 적용

아래 목록은 일부러 비워 두었습니다. 먼저 차종 목록을 출력해서 직접 확인한 뒤 실제 제외값만 입력합니다.

- 제조사 제외: 해당 제조사의 모든 차종을 제외할 때 사용
- 차종 제외: 특정 대표 차종만 제외할 때 사용
- 신고 데이터는 같은 내용처럼 보여도 임의로 중복 제거하지 않습니다. 한 행이 한 신고 기록이기 때문입니다.

In [ ]:
# TODO: 차종 목록을 확인한 뒤 실제 제외값을 직접 입력합니다.
EXCLUDED_MANUFACTURERS = [
    # '제외할 제조사명',
]
EXCLUDED_MODELS = [
    # '정확히 제외할 대표 차종명',
]

def contains_any(series, words):
    if not words:
        return pd.Series(False, index=series.index)
    pattern = '|'.join(re.escape(word) for word in words)
    return series.fillna('').str.contains(pattern, case=False, regex=True)

def apply_exclusions(frame):
    manufacturer_mask = contains_any(frame['manufacturer_name'], EXCLUDED_MANUFACTURERS)
    model_mask = contains_any(frame['model_name'], EXCLUDED_MODELS)
    return frame.loc[~(manufacturer_mask | model_mask)].copy()

defect_processed = apply_exclusions(defect)
recalls_processed = apply_exclusions(recalls)
print('신고 행:', len(defect), '→', len(defect_processed))
print('리콜 행:', len(recalls), '→', len(recalls_processed))

In [ ]:
## 5. 차종 목록을 따로 저장해 검토

model_candidates = (
    pd.concat([
        defect_processed[['manufacturer_name', 'model_name']],
        recalls_processed[['manufacturer_name', 'model_name']],
    ])
    .dropna(subset=['manufacturer_name', 'model_name'])
    .drop_duplicates()
    .sort_values(['manufacturer_name', 'model_name'])
)
display(model_candidates.head(30))
model_candidates.to_csv(MAPPING_DIR / 'model_candidates.csv', index=False, encoding='utf-8-sig')
print('차종 검토 목록 저장:', MAPPING_DIR / 'model_candidates.csv')

In [ ]:
## 6. 전처리 결과 저장

defect_output = PROCESSED_DIR / 'defect_reports.csv'
recall_output = PROCESSED_DIR / 'recalls.csv'

# utf-8-sig로 저장하면 Excel과 VS Code에서 한글을 비교적 안정적으로 열 수 있습니다.
defect_processed.to_csv(defect_output, index=False, encoding='utf-8-sig')
recalls_processed.to_csv(recall_output, index=False, encoding='utf-8-sig')
print('저장 완료')
print(defect_output)
print(recall_output)

In [ ]:
## 7. 저장 결과 검증

defect_check = pd.read_csv(defect_output, encoding='utf-8-sig')
recall_check = pd.read_csv(recall_output, encoding='utf-8-sig')
print('신고 결과 행 수:', len(defect_check))
print('리콜 결과 행 수:', len(recall_check))
print('신고 결측 확인')
display(defect_check.isna().sum())
print('리콜 결측 확인')
display(recall_check.isna().sum())
assert 'source_row_no' in defect_check.columns
assert 'source_row_no' in recall_check.columns
print('검증 완료: 다음 단계는 SQLite 테이블 설계입니다.')

## 다음 단계: SQLite와 Streamlit

전처리 결과를 확인한 뒤에야 DB를 만듭니다. 기존 완성 프로젝트의 다음 파일을 참고해 `scripts`와 `sql`을 직접 작성할 수 있습니다.

- `project_t4/scripts/build_database.py`: CSV를 SQLite 테이블로 넣는 과정
- `project_t4/sql/schema.sql`: 테이블 구조
- `project_t4/sql/queries.sql`: 조회 SQL
- `project_t4/app/streamlit_app.py`: 조회 화면

DB를 만들 때도 신고 데이터와 공식 리콜 데이터를 하나의 의미로 섞지 않습니다. `model_id`를 공통 차량 기준으로 사용하고, 신고의 모델연도와 리콜의 생산기간은 각각 별도로 보여줍니다.